# OptiTrack position plots

This notebook reads every Motive/OptiTrack CSV in `OT-data/`, selects the marker with the best frame coverage for each trial, converts positions to **millimeters**, removes short impulse jumps with a Hampel despike filter, then applies a zero-phase low-pass filter (LPF). Outputs are written to `OT-results/`.

Filtering choice checked for these files:
- The CSV header says `Length Units,Meters`, so values are converted with `meters * 1000 = mm`.
- Small-motion trials are only about 10--13 mm peak-to-peak, so they use a stricter despike step and lower LPF cutoff.
- Big-motion trials are about 230--250 mm peak-to-peak, so they use conservative despiking and a higher LPF cutoff to avoid removing real motion.
- A high-pass filter (HPF) is **not used by default** because HPF removes slow/absolute position content; it is not the right tool for single-frame jumps in position traces. The notebook uses despike + interpolation + LPF instead.

In [1]:
from pathlib import Path
import csv
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.signal import butter, sosfiltfilt
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
    warnings.warn('SciPy is not available; falling back to rolling-average low-pass filtering.')

DATA_DIR = Path('OT-data')
RESULTS_DIR = Path('OT-results')
PLOTS_DIR = RESULTS_DIR / 'position_plots'
FILTERED_DIR = RESULTS_DIR / 'filtered_data'
SUMMARY_CSV = RESULTS_DIR / 'position_summary.csv'
FILTER_NOTES = RESULTS_DIR / 'filter_notes.md'
OVERVIEW_PNG = RESULTS_DIR / 'all_trials_resultant_displacement.png'

# Output folders
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
FILTERED_DIR.mkdir(parents=True, exist_ok=True)

# Filter policy. These are intentionally visible for easy adjustment.
SMALL_MOTION_THRESHOLD_MM = 50.0  # below this, trial is treated as small movement
SMALL_LPF_CUTOFF_HZ = 3.0
BIG_LPF_CUTOFF_HZ = 8.0
SMALL_HAMPEL_WINDOW_S = 0.25
BIG_HAMPEL_WINDOW_S = 0.15
SMALL_HAMPEL_MIN_THRESHOLD_MM = 0.75
BIG_HAMPEL_MIN_THRESHOLD_MM = 5.0
HAMPEL_SIGMAS = 6.0

USE_HPF = False  # checked: not suitable for preserving absolute/big position movement

plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 180
plt.rcParams['axes.grid'] = True

In [2]:
def safe_stem(name: str) -> str:
    """Filename-safe stem."""
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', Path(name).stem).strip('_')


def parse_metadata_and_header(path: Path):
    """Read the 7-line Motive header used by these OptiTrack CSV files."""
    with path.open(newline='', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        header_rows = [next(reader) for _ in range(7)]

    meta = {}
    first = header_rows[0]
    for i in range(0, len(first) - 1, 2):
        key = first[i].strip() if first[i] else ''
        if key:
            meta[key] = first[i + 1].strip()
    return meta, header_rows


def unit_multiplier_to_mm(length_units: str) -> float:
    """Convert Motive length unit labels to millimeters."""
    units = (length_units or '').strip().lower()
    if units.startswith('meter') or units == 'm':
        return 1000.0
    if units.startswith('centimeter') or units == 'cm':
        return 10.0
    if units.startswith('millimeter') or units == 'mm':
        return 1.0
    warnings.warn(f'Unknown Length Units={length_units!r}; assuming meters.')
    return 1000.0


def load_trial_best_marker(path: Path):
    """Load one CSV and choose the marker with the highest valid XYZ coverage."""
    meta, header_rows = parse_metadata_and_header(path)
    marker_names = header_rows[3]

    df = pd.read_csv(path, skiprows=7, header=None, low_memory=False)
    time_s = pd.to_numeric(df.iloc[:, 1], errors='coerce')

    marker_records = []
    for col in range(2, df.shape[1], 3):
        if col + 2 >= df.shape[1]:
            break
        marker_name = marker_names[col].strip() if col < len(marker_names) and marker_names[col].strip() else f'Marker_cols_{col}_{col+2}'
        xyz = df.iloc[:, [col, col + 1, col + 2]].apply(pd.to_numeric, errors='coerce')
        valid = xyz.notna().all(axis=1)
        valid_count = int(valid.sum())
        if valid_count == 0:
            continue
        marker_records.append({
            'marker': marker_name,
            'coverage': float(valid.mean()),
            'valid_count': valid_count,
            'xyz': xyz,
        })

    if not marker_records:
        raise ValueError(f'No valid XYZ marker data found in {path}')

    # Best marker = highest frame coverage, then most samples.
    best = sorted(marker_records, key=lambda r: (r['coverage'], r['valid_count']), reverse=True)[0]

    units = meta.get('Length Units', '')
    to_mm = unit_multiplier_to_mm(units)
    pos_mm_abs = best['xyz'].interpolate(limit_direction='both').to_numpy(dtype=float) * to_mm

    fs = float(meta.get('Export Frame Rate') or meta.get('Capture Frame Rate') or 120.0)
    if not np.isfinite(fs) or fs <= 0:
        fs = 120.0

    time_np = time_s.to_numpy(dtype=float)
    if np.isnan(time_np).all():
        time_np = np.arange(len(pos_mm_abs)) / fs

    # Use the median of the first 0.5 s as a stable origin. This plots movement in mm.
    n0 = max(1, min(len(pos_mm_abs), int(round(0.5 * fs))))
    origin_mm = np.nanmedian(pos_mm_abs[:n0], axis=0)
    pos_mm_rel = pos_mm_abs - origin_mm

    return {
        'path': path,
        'meta': meta,
        'marker': best['marker'],
        'marker_coverage': best['coverage'],
        'fs': fs,
        'time_s': time_np,
        'position_mm_abs': pos_mm_abs,
        'position_mm_rel_raw': pos_mm_rel,
        'origin_mm': origin_mm,
        'all_marker_count': len(marker_records),
    }


def hampel_spike_mask_1d(y, fs, window_s, n_sigmas=6.0, min_threshold_mm=0.75):
    """Return True for impulse-like samples far from the local rolling median."""
    s = pd.Series(np.asarray(y, dtype=float))
    window = max(5, int(round(window_s * fs)))
    if window % 2 == 0:
        window += 1
    min_periods = max(3, window // 3)
    med = s.rolling(window, center=True, min_periods=min_periods).median()
    mad = (s - med).abs().rolling(window, center=True, min_periods=min_periods).median()
    robust_sigma = 1.4826 * mad
    threshold = np.maximum(n_sigmas * robust_sigma.to_numpy(dtype=float), min_threshold_mm)
    diff = (s - med).abs().to_numpy(dtype=float)
    return np.isfinite(diff) & np.isfinite(threshold) & (diff > threshold)


def despike_xyz(position_mm, fs, motion_scale):
    """Replace detected short jumps with interpolation."""
    if motion_scale == 'small':
        window_s = SMALL_HAMPEL_WINDOW_S
        min_thr = SMALL_HAMPEL_MIN_THRESHOLD_MM
    else:
        window_s = BIG_HAMPEL_WINDOW_S
        min_thr = BIG_HAMPEL_MIN_THRESHOLD_MM

    masks = [hampel_spike_mask_1d(position_mm[:, axis], fs, window_s, HAMPEL_SIGMAS, min_thr) for axis in range(3)]
    spike_mask = np.logical_or.reduce(masks)

    cleaned = pd.DataFrame(position_mm, columns=['X_mm', 'Y_mm', 'Z_mm'])
    cleaned.loc[spike_mask, :] = np.nan
    cleaned = cleaned.interpolate(limit_direction='both')
    return cleaned.to_numpy(dtype=float), spike_mask, {
        'hampel_window_s': window_s,
        'hampel_min_threshold_mm': min_thr,
        'hampel_sigmas': HAMPEL_SIGMAS,
    }


def lowpass_xyz(position_mm, fs, cutoff_hz):
    """Zero-phase low-pass filter. Falls back to centered rolling mean if SciPy is missing."""
    nyquist = fs / 2.0
    cutoff_hz = min(float(cutoff_hz), 0.45 * fs)
    if SCIPY_AVAILABLE and len(position_mm) > 30 and cutoff_hz < nyquist:
        sos = butter(4, cutoff_hz / nyquist, btype='lowpass', output='sos')
        return sosfiltfilt(sos, position_mm, axis=0)

    # Fallback rough equivalent: rolling average window about 1/cutoff seconds.
    window = max(3, int(round(fs / max(cutoff_hz, 0.1))))
    if window % 2 == 0:
        window += 1
    return pd.DataFrame(position_mm).rolling(window, center=True, min_periods=1).mean().to_numpy(dtype=float)


def process_trial(path: Path):
    trial = load_trial_best_marker(path)
    raw = trial['position_mm_rel_raw']
    time_s = trial['time_s']
    fs = trial['fs']

    span_xyz = np.nanmax(raw, axis=0) - np.nanmin(raw, axis=0)
    resultant_span = float(np.linalg.norm(span_xyz))
    max_axis_span = float(np.nanmax(span_xyz))
    motion_scale = 'small' if max_axis_span < SMALL_MOTION_THRESHOLD_MM else 'big'
    cutoff_hz = SMALL_LPF_CUTOFF_HZ if motion_scale == 'small' else BIG_LPF_CUTOFF_HZ

    despiked, spike_mask, despike_params = despike_xyz(raw, fs, motion_scale)
    filtered = lowpass_xyz(despiked, fs, cutoff_hz)

    step_mm = np.linalg.norm(np.diff(raw, axis=0), axis=1)
    filt_step_mm = np.linalg.norm(np.diff(filtered, axis=0), axis=1)

    out_stem = safe_stem(path.name)
    filtered_csv = FILTERED_DIR / f'{out_stem}_filtered_position_mm.csv'
    out_df = pd.DataFrame({
        'time_s': time_s,
        'raw_X_mm': raw[:, 0],
        'raw_Y_mm': raw[:, 1],
        'raw_Z_mm': raw[:, 2],
        'filtered_X_mm': filtered[:, 0],
        'filtered_Y_mm': filtered[:, 1],
        'filtered_Z_mm': filtered[:, 2],
        'spike_removed': spike_mask,
    })
    out_df.to_csv(filtered_csv, index=False)

    # Time-series position plot.
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    labels = ['X', 'Y', 'Z']
    colors = ['tab:blue', 'tab:orange', 'tab:green']
    for axis, ax in enumerate(axes):
        ax.plot(time_s, raw[:, axis], color='0.65', linewidth=0.7, alpha=0.55, label='raw')
        ax.plot(time_s, filtered[:, axis], color=colors[axis], linewidth=1.2, label=f'filtered ({cutoff_hz:g} Hz LPF)')
        if spike_mask.any():
            ax.scatter(time_s[spike_mask], raw[spike_mask, axis], s=8, c='crimson', alpha=0.65, label='despiked samples' if axis == 0 else None)
        ax.set_ylabel(f'{labels[axis]} (mm)')
        ax.legend(loc='upper right', fontsize=8)
    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(f'{path.stem}\nBest marker: {trial["marker"]} | {motion_scale} motion | origin = first 0.5 s median')
    fig.tight_layout(rect=[0, 0.02, 1, 0.96])
    position_png = PLOTS_DIR / f'{out_stem}_position_mm.png'
    fig.savefig(position_png, bbox_inches='tight')
    plt.close(fig)

    # 2D X-Z path plot. In this dataset X/Z carry most of the large movements.
    fig, ax = plt.subplots(figsize=(7.5, 7))
    ax.plot(raw[:, 0], raw[:, 2], color='0.75', linewidth=0.7, alpha=0.7, label='raw X-Z path')
    sc = ax.scatter(filtered[:, 0], filtered[:, 2], c=time_s, s=4, cmap='viridis', label='filtered path')
    ax.set_xlabel('X displacement (mm)')
    ax.set_ylabel('Z displacement (mm)')
    ax.set_title(f'{path.stem}\nFiltered X-Z path')
    ax.axis('equal')
    fig.colorbar(sc, ax=ax, label='Time (s)')
    ax.legend(loc='best', fontsize=8)
    path_png = PLOTS_DIR / f'{out_stem}_path_XZ_mm.png'
    fig.savefig(path_png, bbox_inches='tight')
    plt.close(fig)

    return {
        'file': path.name,
        'marker': trial['marker'],
        'marker_coverage_pct': 100.0 * trial['marker_coverage'],
        'frames': len(raw),
        'duration_s': float(np.nanmax(time_s) - np.nanmin(time_s)),
        'length_units_in_file': trial['meta'].get('Length Units', ''),
        'converted_to': 'mm',
        'motion_scale': motion_scale,
        'span_X_mm_raw': float(span_xyz[0]),
        'span_Y_mm_raw': float(span_xyz[1]),
        'span_Z_mm_raw': float(span_xyz[2]),
        'resultant_span_mm_raw': resultant_span,
        'raw_step_p99_mm_per_frame': float(np.nanpercentile(step_mm, 99)) if len(step_mm) else np.nan,
        'raw_step_max_mm_per_frame': float(np.nanmax(step_mm)) if len(step_mm) else np.nan,
        'filtered_step_p99_mm_per_frame': float(np.nanpercentile(filt_step_mm, 99)) if len(filt_step_mm) else np.nan,
        'filtered_step_max_mm_per_frame': float(np.nanmax(filt_step_mm)) if len(filt_step_mm) else np.nan,
        'spikes_removed': int(spike_mask.sum()),
        'spikes_removed_pct': float(100.0 * spike_mask.mean()),
        'lpf_cutoff_hz': cutoff_hz,
        'hpf_used': USE_HPF,
        **despike_params,
        'position_plot': str(position_png),
        'path_plot': str(path_png),
        'filtered_csv': str(filtered_csv),
    }, time_s, filtered

In [3]:
csv_files = sorted(DATA_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No CSV files found in {DATA_DIR.resolve()}')

summary_rows = []
overview_series = []

for path in csv_files:
    row, time_s, filtered = process_trial(path)
    summary_rows.append(row)
    resultant = np.linalg.norm(filtered - filtered[0, :], axis=1)
    overview_series.append((path.stem, time_s, resultant, row['motion_scale']))

summary = pd.DataFrame(summary_rows)
summary.to_csv(SUMMARY_CSV, index=False)

# Overview plot: resultant displacement for every trial.
fig, ax = plt.subplots(figsize=(12, 7))
for label, time_s, resultant, motion_scale in overview_series:
    lw = 1.25 if motion_scale == 'big' else 1.0
    alpha = 0.85 if motion_scale == 'big' else 0.55
    ax.plot(time_s, resultant, linewidth=lw, alpha=alpha, label=label.replace('Take 2026-09-01 ', ''))
ax.set_xlabel('Time (s)')
ax.set_ylabel('Resultant displacement from start (mm)')
ax.set_title('All OptiTrack trials ? filtered resultant displacement')
ax.legend(fontsize=7, ncol=2, loc='best')
fig.tight_layout()
fig.savefig(OVERVIEW_PNG, bbox_inches='tight')
plt.close(fig)

small_count = int((summary['motion_scale'] == 'small').sum())
big_count = int((summary['motion_scale'] == 'big').sum())
notes = f"""# OptiTrack filter notes

Processed {len(summary)} CSV files from `{DATA_DIR}` into `{RESULTS_DIR}`.

## Unit conversion
The files report `Length Units` in their Motive header. These runs report meters, so the notebook converts values to millimeters for plotting movement.

## Filter decision
- Small-motion trials: {small_count}; LPF cutoff = {SMALL_LPF_CUTOFF_HZ} Hz; Hampel despike minimum threshold = {SMALL_HAMPEL_MIN_THRESHOLD_MM} mm.
- Big-motion trials: {big_count}; LPF cutoff = {BIG_LPF_CUTOFF_HZ} Hz; Hampel despike minimum threshold = {BIG_HAMPEL_MIN_THRESHOLD_MM} mm.
- HPF used: {USE_HPF}.

I did not apply HPF because these are position traces and HPF would remove slow/absolute position motion, especially the real large X/Z displacement in the big-motion trials. For jump cleanup, the notebook removes impulse-like local outliers first, interpolates across those samples, then applies LPF.

## Main outputs
- Position plots: `{PLOTS_DIR}`
- Filtered per-trial CSV files: `{FILTERED_DIR}`
- Summary CSV: `{SUMMARY_CSV}`
- Overview plot: `{OVERVIEW_PNG}`
"""
FILTER_NOTES.write_text(notes, encoding='utf-8')

print(f'Processed {len(summary)} CSV files')
print(f'Wrote results to: {RESULTS_DIR.resolve()}')
print(f'Position plots: {PLOTS_DIR.resolve()}')
print(f'Summary: {SUMMARY_CSV.resolve()}')
summary[['file', 'marker', 'motion_scale', 'resultant_span_mm_raw', 'spikes_removed', 'lpf_cutoff_hz', 'hpf_used']]

Processed 21 CSV files
Wrote results to: C:\dev\Parallel_Heptics\validetion\optitrack\OT-results
Position plots: C:\dev\Parallel_Heptics\validetion\optitrack\OT-results\position_plots
Summary: C:\dev\Parallel_Heptics\validetion\optitrack\OT-results\position_summary.csv


                           file      marker motion_scale  resultant_span_mm_raw  spikes_removed  lpf_cutoff_hz  hpf_used
Take 2026-09-01 06.38.57 PM.csv Marker_3346        small              12.572592             223            3.0     False
Take 2026-09-01 06.42.02 PM.csv Marker_3346        small              12.711118             268            3.0     False
Take 2026-09-01 06.45.00 PM.csv Marker_3346        small              12.647156             340            3.0     False
Take 2026-09-01 06.48.04 PM.csv Marker_3346        small              12.706747             624            3.0     False
Take 2026-09-01 06.55.48 PM.csv Marker_3456        small              13.337757             463            3.0     False
Take 2026-09-01 07.22.32 PM.csv  Marker_785          big             248.425916               1            8.0     False
Take 2026-09-01 07.24.02 PM.csv  Marker_785          big             232.715989               0            8.0     False
Take 2026-09-01 07.25.01 PM.csv 

## Interpretation notes

Use the per-trial `*_position_mm.png` plots to compare raw vs filtered X/Y/Z movement. Red points mark samples classified as short impulse jumps and replaced by interpolation before LPF.

For these files, HPF is intentionally off. If later files show true slow drift unrelated to the test motion, add drift-removal explicitly (for example, subtract a baseline/trend from known stationary periods) rather than blindly high-pass filtering all position data.